# 🧱 Modul 03 Hands-on: CNN Architecture Deep Dive

Pada hands-on ini, Anda akan membedah arsitektur CNN modern:
1. **The Working Blackbox**: Membangun dan melatih model CNN modern pada data visual.
2. **Anatomical Dissection**: Menghitung Receptive Field dan memvisualisasikan Feature Maps antar layer.
3. **The Sabotage Lab**: Degradasi Jaringan Dalam! Mengamati model 14-layer TANPA skip connection yang gagal belajar vs ResNet DENGAN skip connection ($y = F(x) + x$) yang konvergen mulus.
4. **Depthwise Separable Convolution**: Membuktikan efisiensi komputasi arsitektur MobileNet.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Device aktif: {device}")

## 1. Menyiapkan Dataset Sintetis 10-Kelas

In [ ]:
# 500 gambar sintetis 3x32x32
torch.manual_seed(42)
X_train = torch.randn(400, 3, 32, 32).to(device)
y_train = torch.randint(0, 10, (400,)).to(device)
X_val = torch.randn(100, 3, 32, 32).to(device)
y_val = torch.randint(0, 10, (100,)).to(device)
print(f"Data train: {X_train.shape}, Labels: {y_train.shape}")

## 2. The Sabotage Lab: Degradation Problem in Deep Plain Networks
Sebelum ResNet (2015), menumpuk 15+ layer konvolusi justru membuat akurasi anjlok karena hilangnya aliran gradien.
Mari kita adu:
- **Model A (Plain 10-Layer)**: Menumpuk Conv2d tanpa skip connection.
- **Model B (ResNet 10-Layer)**: Memakai Residual Block $y = F(x) + x$.

In [ ]:
# Model A: Deep Plain Network (TANPA Skip Connection)
class PlainBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))  # TANPA + x
        return out

# Model B: Residual Block (DENGAN Skip Connection)
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)  # <--- KUNCI RESNET: + x!
        return out

class DeepNet(nn.Module):
    def __init__(self, block_cls, num_blocks=5, channels=32):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, channels, 3, padding=1), nn.BatchNorm2d(channels), nn.ReLU())
        self.blocks = nn.Sequential(*[block_cls(channels) for _ in range(num_blocks)])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channels, 10)
    def forward(self, x):
        return self.fc(self.pool(self.blocks(self.stem(x))).flatten(1))

model_plain = DeepNet(PlainBlock).to(device)
model_resnet = DeepNet(ResBlock).to(device)
print("Model Plain & ResNet berhasil dibuat dengan parameter identik!")

## 3. Adu Konvergensi: Plain Net vs ResNet (20 Epochs)

In [ ]:
def train_model(model, lr=0.01, epochs=20):
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    losses = []
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(X_train)
        l = crit(out, y_train)
        l.backward()
        opt.step()
        losses.append(l.item())
    return losses

plain_losses = train_model(model_plain)
resnet_losses = train_model(model_resnet)

plt.figure(figsize=(9, 5))
plt.plot(plain_losses, "r--", label=f"Plain Net (Tanpa Skip) -> Akhir: {plain_losses[-1]:.4f}", linewidth=2)
plt.plot(resnet_losses, "g-", label=f"ResNet (Dengan Skip)  -> Akhir: {resnet_losses[-1]:.4f}", linewidth=2.5)
plt.title("The Sabotage Lab: Plain Net Degradation vs ResNet Shortcut", fontsize=13, fontweight="bold")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(fontsize=11); plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 4. Depthwise Separable Convolution (MobileNet)
Membandingkan jumlah parameter Conv2d standar vs Depthwise Separable Conv.

In [ ]:
in_c, out_c, k = 64, 128, 3
# 1. Standar Conv2d
standard_conv = nn.Conv2d(in_c, out_c, kernel_size=k, padding=1, bias=False)
params_std = sum(p.numel() for p in standard_conv.parameters())

# 2. Depthwise Separable: Depthwise (groups=in_c) + Pointwise (1x1)
depthwise = nn.Conv2d(in_c, in_c, kernel_size=k, padding=1, groups=in_c, bias=False)
pointwise = nn.Conv2d(in_c, out_c, kernel_size=1, bias=False)
params_ds = sum(p.numel() for p in depthwise.parameters()) + sum(p.numel() for p in pointwise.parameters())

print(f"Parameter Standar Conv 3x3 : {params_std:,}")
print(f"Parameter Depthwise Separable: {params_ds:,}")
print(f"Penghematan: {((params_std - params_ds) / params_std) * 100:.1f}% lebih sedikit parameter!")